# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: fetch latest code</h2>
            <span style="color:#f71;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull and merge your changes as needed</a>. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/><br/>
            After you've pulled the code, from the llm_engineering directory, in a Cursor Terminal, run:<br/>
            <code>uv sync</code><br/>
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
from groq import Groq
import subprocess
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY') 
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

# if openai_api_key:
#     print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
# else:
#     print("OpenAI API Key not set")
    
# if anthropic_api_key:
#     print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
# else:
#     print("Anthropic API Key not set (and this is optional)")

# if google_api_key:
#     print(f"Google API Key exists and begins {google_api_key[:2]}")
# else:
#     print("Google API Key not set (and this is optional)")

# if grok_api_key:
#     print(f"Grok API Key exists and begins {grok_api_key[:4]}")
# else:
#     print("Grok API Key not set (and this is optional)")
    
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

Groq API Key exists and begins gsk_


In [3]:
# Connect to client libraries

# openai = OpenAI()

groq = Groq()

# anthropic_url = "https://api.anthropic.com/v1/"
# gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
# grok_url = "https://api.x.ai/v1"

# anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
# gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
# grok = OpenAI(api_key=grok_api_key, base_url=grok_url)


In [4]:
OPENAI_MODEL = "gpt-5"
CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
GROK_MODEL = "grok-4"
GEMINI_MODEL = "gemini-2.5-pro"
GROQ_MODEL = "openai/gpt-oss-120b"

# Want to keep costs ultra-low? Uncomment these lines:

# OPENAI_MODEL = "gpt-5-nano"
# CLAUDE_MODEL = "claude-3-5-haiku-latest"
# GROK_MODEL = "grok-4-fast-non-reasoning"
# GEMINI_MODEL = "gemini-2.5-flash-lite"

## PLEASE NOTE:

We will be writing a solution to convert Python into efficient, optimized C++ code for your machine, which can be compiled to native machine code and executed.

It is not necessary for you to execute the code yourself - that's not the point of the exercise!

But if you would like to (because it's satisfying!) then I'm including the steps here. Very optional!

As an alternative, I'll also show you a website where you can run the C++ code.

In [5]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '25.6.0',
  'version': 'Darwin Kernel Version 25.6.0: Fri Jul 31 19:18:49 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T6000',
  'kernel': '25.6.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin25.6.0'},
 'package_managers': ['xcode-select (CLT)'],
 'cpu': {'brand': 'Apple M1 Pro',
  'cores_logical': 8,
  'cores_physical': 8,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'g++': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'clang': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

In [8]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = groq.chat.completions.create(model=GROQ_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

### Do you already have a C++ compiler?

Your system information shows the following compilers are available:

```
gcc      → Apple clang version 21.0.0 (clang‑2100.1.1.101)
g++      → Apple clang version 21.0.0 (clang‑2100.1.1.101)
clang    → Apple clang version 21.0.0 (clang‑2100.1.1.101)
```

That means the **Xcode Command‑Line Tools** (CLT) are installed and you already have a working C++ compiler (`clang++`, which is also invoked as `g++`).  
No additional installation is required to compile a single `main.cpp` file.

> **If you ever see “command not found: clang”** you can install the CLT with:

```bash
xcode-select --install
```

After the GUI installer finishes, `clang`/`clang++` will be on your `$PATH`.

---

## Fastest‑possible compile‑and‑run from Python

Below is a minimal, high‑performance compile command and the corresponding run command you can feed directly to `subprocess.run`.  
The commands are given as **lists** (recommended for `subprocess.run` with `shell=False`).

```python
import subprocess
from pathlib import Path

# -------------------------------------------------
# 1️⃣  Compile   (fastest typical release build)
# -------------------------------------------------
#   -O3      → maximum optimisation
#   -march=native → generate code tuned for the host CPU (Apple M1 Pro)
#   -std=c++20    → modern language features (you can lower this if you wish)
#   -Wall -Wextra → optional, helpful warnings
#   -DNDEBUG      → disables assert() in release builds
#   -o main       → output executable named "main"
compile_command = [
    "clang++",                # the C++ front‑end (alias of g++)
    "-O3",
    "-march=native",
    "-std=c++20",
    "-Wall",
    "-Wextra",
    "-DNDEBUG",
    "main.cpp",               # source file in the current working directory
    "-o", "main"              # output binary
]

compile_result = subprocess.run(
    compile_command,
    check=True,               # raise CalledProcessError on failure
    text=True,
    capture_output=True       # you can inspect compile_result.stdout / stderr
)

# -------------------------------------------------
# 2️⃣  Run the produced binary
# -------------------------------------------------
# On macOS the executable is in the current directory, so we prefix it with "./"
run_command = ["./main"]

run_result = subprocess.run(
    run_command,
    check=True,
    text=True,
    capture_output=True
)

print("Program output:")
print(run_result.stdout)
```

### Why these flags?

| Flag | Reason |
|------|--------|
| `-O3` | Highest level of optimisation that the compiler provides. |
| `-march=native` | Tells clang to emit instructions tailored for the exact CPU you’re running on (Apple M1 Pro → ARM64). This gives the best runtime speed without sacrificing portability on your own machine. |
| `-std=c++20` | Gives you the latest language features while still being well‑supported by Apple clang. Change to `c++17` or `c++14` if your code requires an older standard. |
| `-Wall -Wextra` | Enables a broad set of warnings – useful while developing, harmless in a release build. |
| `-DNDEBUG` | Disables the `assert()` macro, which can shave a few cycles in hot loops. |
| `-o main` | Produces a cleanly‑named executable (`main`). |

If you prefer an even more “bare‑metal” binary (no debug symbols, smallest size) you could add:

```bash
"-flto",               # link‑time optimisation
"-s"                   # strip all symbols from the final executable
```

Resulting compile list:

```python
compile_command = [
    "clang++", "-O3", "-march=native", "-std=c++20",
    "-Wall", "-Wextra", "-DNDEBUG",
    "-flto", "-s",
    "main.cpp", "-o", "main"
]
```

---

## Quick sanity check (run in a terminal)

If you just want to test that everything works without Python, you can type:

```bash
# 1️⃣ Compile
clang++ -O3 -march=native -std=c++20 -Wall -Wextra -DNDEBUG main.cpp -o main

# 2️⃣ Execute
./main
```

You should see whatever `main.cpp` prints to `stdout`.

---

### TL;DR

* **No extra installation needed** – you already have `clang++` via Xcode CLT.
* Use the compile command list shown above (`clang++ … -o main`) and run it with `./main` from Python’s `subprocess.run`.
* The provided flags give you the fastest typical release build on your Apple M1 Pro.

## If you need to install something

If you would like to, please follow GPTs instructions! Then rerun the analysis afterwards (you might need to Restart the notebook) to confirm you're set.

You should now be equipped with the command to compile the code, and the command to run it!

Enter that in the cell below:

In [9]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

## And now, on with the main task

In [10]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [11]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [12]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [26]:
def port(client, model, python):
    request = {
        "model": model,
        "messages": messages_for(python),
        "max_completion_tokens": 8192,
        "reasoning_effort": "medium",
    }

    response = client.chat.completions.create(**request)

    message = response.choices[0].message

    print("Finish reason:", response.choices[0].finish_reason)
    print("Content:", repr(message.content))
    print("Reasoning:", repr(message.reasoning))

    if not message.content:
        raise RuntimeError("Model returned empty content")

    reply = message.content
    reply = reply.replace("```cpp", "").replace("```", "").strip()

    write_output(reply)

In [19]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [20]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [27]:
run_python(pi)

Result: 3.141592656089
Execution Time: 26.191755 seconds


In [28]:
port(groq, GROQ_MODEL, pi)

Finish reason: stop
Content: '```cpp\n#include <bits/stdc++.h>\nusing namespace std;\n\ninline double calculate(uint64_t iterations, double p1, double p2) {\n    double result = 1.0;\n    for (uint64_t i = 1; i <= iterations; ++i) {\n        double j = i * p1 - p2;\n        result -= 1.0 / j;\n        j = i * p1 + p2;\n        result += 1.0 / j;\n    }\n    return result;\n}\n\nint main() {\n    ios::sync_with_stdio(false);\n    cin.tie(nullptr);\n\n    auto start = chrono::high_resolution_clock::now();\n\n    double res = calculate(200\'000\'000ULL, 4.0, 1.0) * 4.0;\n\n    auto end = chrono::high_resolution_clock::now();\n    double elapsed = chrono::duration<double>(end - start).count();\n\n    cout.setf(ios::fixed);\n    cout << setprecision(12) << "Result: " << res << "\\n";\n    cout << setprecision(6)  << "Execution Time: " << elapsed << " seconds\\n";\n    return 0;\n}\n```\n'
Reasoning: "We need to write C++ program that reproduces output: compute same result as Python double (

# Compiling C++ and executing

This next cell contains the command to compile a C++ file based on the instructions from GPT.

Again, it's not crucial to do this step if you don't wish to!

OR alternatively: student Sandeep K.G. points out that you can run Python and C++ code online to test it out that way. Thank you Sandeep!  
> Not an exact comparison but you can still get the idea of performance difference.  
> For example here: https://www.programiz.com/cpp-programming/online-compiler/

In [ ]:
# Use the commands from GPT 5

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [ ]:
compile_and_run()

In [ ]:
19.178207/0.082168

## OK let's try the other contenders!

In [ ]:
port(anthropic, CLAUDE_MODEL, pi)
compile_and_run()

In [ ]:
port(grok, GROK_MODEL, pi)
compile_and_run()

In [ ]:
port(gemini, GEMINI_MODEL, pi)
compile_and_run()


In [ ]:
print(f"""
In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: {19.178207/0.104241:.0f}X speedup
3rd place: GPT-5: {19.178207/0.082168:.0f}X speedup
2nd place: Grok 4: {19.178207/0.018092:.0f}X speedup
1st place: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup
""")